In [ ]:
from fastai.collab import *from fastai.metrics import *from fastai.data.external import *

In [ ]:
import re, sys, osimport loggingfrom pathlib import Pathfrom datetime import datetimeimport torchimport torch_directmltorch._logging.set_logs(all=logging.WARNING)

In [ ]:
print("="*80)print("DIRECTML FULL DEBUG LOG STARTED")print(f"torch version: {torch_directml.torch.__version__}")print(f"Device: {torch_directml.device()}")print(f"GPU: {torch_directml.device_name(0)}")print("="*80)

In [ ]:
# 1. Detect DirectML device (no global default!)dml = torch_directml.device()

## Local dataset cache

To avoid re-downloading the same data across runs, we save the dataset into a local `data/` folder inside the repo. If the data already exists, `untar_data` will reuse it rather than downloading again.

In [ ]:
local_data_path = Path('data')local_data_path.mkdir(parents=True, exist_ok=True)path = untar_data(URLs.ML_SAMPLE, dest=local_data_path)print(f'FastAI dataset path: {path}')

In [ ]:
dls = CollabDataLoaders.from_csv(    path / 'ratings.csv', bs=64, device=dml,    verbose=True, num_workers=0)

In [ ]:
# For example, accuracy within a thresholddef rating_accuracy(inp, targ):    """Percentage of predictions within 0.5 of actual rating"""    return ((inp - targ).abs() < 0.5).float().mean()

In [ ]:
learn = collab_learner(    dls, y_range=(0.5,5.5), metrics=[rmse]).to_fp16(enabled=False)learn.to(dml)""

In [ ]:
# learn.fit_one_cycle instead of learn.fine_tune# because we do not have a pretrained model for tabular datalearn.fit_one_cycle(20, lr_max=5e-3)

In [ ]:
learn.show_results()

## Benchmark: CPU vs DirectML Adreno GPU

Now we run a small benchmark for the same collab model on both CPU-only and the DirectML Adreno GPU. This lets us compare training runtime and capture where work is executed.

In [ ]:
import timeimport pandas as pdresults = []# GPU benchmark using DirectMLprint('GPU device:', torch_directml.device_name(0))print('DirectML available:', torch_directml.is_available())gpu_device = torch_directml.device()dls_gpu = CollabDataLoaders.from_csv(    path / 'ratings.csv', bs=64, device=gpu_device,    verbose=True, num_workers=0)learn_gpu = collab_learner(    dls_gpu, y_range=(0.5,5.5), metrics=[rmse]).to_fp16(enabled=False)learn_gpu.to(gpu_device)n_epochs = 5start = time.perf_counter()learn_gpu.fit_one_cycle(n_epochs, lr_max=5e-3)gpu_time = time.perf_counter() - startprint(f'GPU DirectML run time: {gpu_time:.2f}s')results.append({    'setup': 'DirectML Adreno GPU',    'epochs': n_epochs,    'duration_s': gpu_time,    'notes': 'DirectML device; some optimizer ops may fallback to CPU'})

In [ ]:
# CPU-only benchmarkcpu_device = torch.device('cpu')dls_cpu = CollabDataLoaders.from_csv(    path / 'ratings.csv', bs=64, device=cpu_device,    verbose=True, num_workers=0)learn_cpu = collab_learner(    dls_cpu, y_range=(0.5,5.5), metrics=[rmse]).to_fp16(enabled=False)learn_cpu.to(cpu_device)start = time.perf_counter()learn_cpu.fit_one_cycle(n_epochs, lr_max=5e-3)cpu_time = time.perf_counter() - startprint(f'CPU run time: {cpu_time:.2f}s')results.append({    'setup': 'CPU-only',    'epochs': n_epochs,    'duration_s': cpu_time,    'notes': 'CPU-only baseline'})

In [ ]:
df = pd.DataFrame(results)df['duration_min'] = df['duration_s'] / 60df

## Summary

This section compares the CPU-only baseline against the DirectML Adreno GPU setup. The left column is the CPU run and the right column is the GPU run. The metrics include total training time and any notes about fallback behavior.

- `DirectML Adreno GPU`: uses `torch-directml` and your Adreno device. Some unsupported backend operators may fall back to CPU.
- `CPU-only`: uses the same FastAI collab model and data entirely on CPU.

Use this comparison to judge whether the GPU path provides a meaningful speed advantage for your workflow.